In [14]:
puts `head -2 ./maps/genes.map`

sourceid,label,geneid,protein,recommended_full,taxon
ENSG00000012779,ENSG00000012779,http://purl.uniprot.org/geneid/240,http://purl.uniprot.org/uniprot/P09917,Polyunsaturated fatty acid 5-lipoxygenase,http://purl.uniprot.org/taxonomy/9606


In [15]:
puts `head -2 ./maps/diseases.map`

source,mondo,prefname
EFO_0000174,http://purl.obolibrary.org/obo/MONDO_0012817,Ewing sarcoma


In [20]:
puts `grep Orphanet ./maps/diseases.map | head -20`

Orphanet_95,http://purl.obolibrary.org/obo/MONDO_0100339,Friedreich ataxia
Orphanet_950,http://purl.obolibrary.org/obo/MONDO_0019797,Acrodysostosis
Orphanet_95159,http://purl.obolibrary.org/obo/MONDO_0019799,Hepatoerythropoietic porphyria
Orphanet_952,http://purl.obolibrary.org/obo/MONDO_0008673,"Acrofacial dysostosis, Weyers type"
Orphanet_95232,http://purl.obolibrary.org/obo/MONDO_0011830,Lissencephaly due to LIS1 mutation
Orphanet_95428,http://purl.obolibrary.org/obo/MONDO_0012635,COG8-CDG
Orphanet_95429,http://purl.obolibrary.org/obo/MONDO_0019803,Angioma serpiginosum
Orphanet_95430,http://purl.obolibrary.org/obo/MONDO_0019804,Congenital tracheomalacia
Orphanet_95433,http://purl.obolibrary.org/obo/MONDO_0013931,Autosomal recessive spinocerebellar ataxia-blindness-deafness syndrome
Orphanet_95434,http://purl.obolibrary.org/obo/MONDO_0011811,Autosomal recessive cerebellar ataxia-movement disorder syndrome
Orphanet_95496,http://purl.obolibrary.org/obo/MONDO_0019828,Pituitary stalk int

In [16]:
puts `head -2 ./rawdata/disease-gene.csv`

"source","source_type","target","target_type"
"DOID_7551","disease","ENSG00000058085","gene"


In [19]:
require 'linkeddata'
require 'rdf/nquads'
require 'csv'

e = File.open('./graph/disease-gene-errors.txt', 'w') 

# Define namespaces
SIMPATHIC = RDF::Vocabulary.new('urn:simpathic:')
RDFS = RDF::Vocabulary.new('http://www.w3.org/2000/01/rdf-schema#')
# Read input files
disease_mappings = CSV.read('./maps/diseases.map', headers: true)
gene_mappings = CSV.read('./maps/genes.map', headers: true)
failures = {}


# refresh
f = File.open('./graph/disease-gene.nq.large', 'w')
f.close

recordcount = 0
CSV.foreach('./rawdata/disease-gene.csv', col_sep: ",", quote_char: '"', 
  liberal_parsing: true, headers: true) do |row|
# "source","source_type","target","target_type"
# "DOID_7551","disease","ENSG00000058085","gene"
  disease_id = row['source']
  gene_id = row['target']
  score = 1
  #evidence = ""

#   warn "searching for #{disease_id}"
  disease = disease_mappings.find { |d| d['source'] == disease_id }
  gene = gene_mappings.find { |d| d['sourceid'] == gene_id }
  
  unless disease
    next if failures[disease_id]
    failures[disease_id] = 1
    warn "disease lookup failed #{disease_id}"
    e.write "disease lookup failed #{disease_id}\n"
    next
  end
  unless gene
    next if failures[gene_id]
    failures[gene_id] = 1
    warn "gene lookup failed #{gene_id}"
    e.write "gene lookup failed #{gene_id}\n"
    next
  end
  
  # Extract relevant IDs and labels
# source,mondo,prefname
# Orphanet:100032,MONDO:0968955,Hypocalcified amelogenesis imperfecta
  mondo_uri = RDF::URI.new(disease['mondo'])
  mondo_type = RDF::URI.new("https://bioportal.bioontology.org/ontologies/MONDO")
  mondo_core_type = RDF::URI.new("https://w3id.org/biolink/vocab/Disease")
  mondo_label =  RDF::Literal.new("MONDO Term")
#   orphanet = RDF::URI.new(disease['orpha'])
  disease_label = RDF::Literal.new(disease['prefname'])
  original_disease = RDF::Literal.new(disease['source'])
  

#   sourceid,label,geneid,protein,recommended_full,taxon
#   ENSG00000091831,ENSG00000091831,http://purl.uniprot.org/geneid/2099,http://purl.uniprot.org/uniprot/P03372,Estrogen receptor,http://purl.uniprot.org/taxonomy/9606
  gene_uri = RDF::URI.new(gene['geneid'])
  gene_type = RDF::URI.new("http://edamontology.org/data_1027")
  gene_label =  RDF::Literal.new("NCBI/UniProt Gene Identifier")
  gene_core_type = RDF::URI.new("https://w3id.org/biolink/vocab/Gene")
  gene_label = RDF::Literal.new(gene['label'])

  
  protein_uri = RDF::URI.new(gene['protein'])
  protein_type = RDF::URI.new("http://edamontology.org/data_2291")
  protein_label =  RDF::Literal.new("UniProt Identifier")
  protein_core_type = RDF::URI.new("https://w3id.org/biolink/vocab/Protein")
  human_protein_label = RDF::Literal.new(gene['recommended_full'])
    
  taxon = RDF::URI.new(gene['taxon'])
  
  # Create context URI
  context_uri = RDF::URI.new("urn:simpathic:context:#{disease_id}_#{gene_id}")
  general_context = RDF::URI.new("urn:simpathic:context:all_metadata")
  
  # Create RDF repository (need to do this each time, since there are hundreds of thousands of lines, and the graph gets too big for memory)
  graph = RDF::Repository.new


  # Add quads to graph using RDF::Statement
  graph << RDF::Statement.new(mondo_uri, SIMPATHIC['associated-with'], protein_uri, graph_name: context_uri)
  graph << RDF::Statement.new(protein_uri, SIMPATHIC['associated-with'], mondo_uri, graph_name: context_uri)

    
  graph << RDF::Statement.new(mondo_uri, RDFS.label, disease_label, graph_name: context_uri)
  graph << RDF::Statement.new(mondo_uri, RDF.type, mondo_type, graph_name: context_uri)
  graph << RDF::Statement.new(mondo_uri, RDF.type, mondo_core_type, graph_name: context_uri)
  graph << RDF::Statement.new(mondo_type, RDFS.label, mondo_label, graph_name: context_uri)
#   graph << RDF::Statement.new(mondo_uri, SIMPATHIC['orphanet'], orphanet, graph_name: context_uri)
  graph << RDF::Statement.new(mondo_uri, SIMPATHIC['original-id'], original_disease, graph_name: context_uri)
  
  graph << RDF::Statement.new(gene_uri,  RDFS.label,       gene_label , graph_name: context_uri)
  graph << RDF::Statement.new(gene_uri,  RDF.type,         gene_type, graph_name: context_uri)
  graph << RDF::Statement.new(gene_uri,  RDF.type,         gene_core_type, graph_name: context_uri)
  graph << RDF::Statement.new(gene_type, RDFS.label,       RDF::Literal.new("NCBI Gene"), graph_name: context_uri)
  graph << RDF::Statement.new(gene_core_type, RDFS.label,  RDF::Literal.new("Gene"), graph_name: context_uri)
  graph << RDF::Statement.new(gene_uri,  SIMPATHIC['original-id'], RDF::Literal.new("#{gene_id}"), graph_name: context_uri)
  graph << RDF::Statement.new(gene_uri,  SIMPATHIC['in-taxon'], taxon, graph_name: context_uri)
    
  graph << RDF::Statement.new(mondo_uri, SIMPATHIC['associated-with'], gene_uri, graph_name: context_uri)
  graph << RDF::Statement.new(gene_uri, SIMPATHIC['associated-with'], mondo_uri, graph_name: context_uri)

  
  graph << RDF::Statement.new(protein_uri,  RDFS.label,       human_protein_label , graph_name: context_uri)
  graph << RDF::Statement.new(protein_uri,  RDF.type,         protein_type, graph_name: context_uri)
  graph << RDF::Statement.new(protein_uri,  RDF.type,         protein_core_type, graph_name: context_uri)
  graph << RDF::Statement.new(protein_type, RDFS.label,       RDF::Literal.new("UniProt"), graph_name: context_uri)
  graph << RDF::Statement.new(protein_core_type, RDFS.label,  RDF::Literal.new("Protein"), graph_name: context_uri)
  graph << RDF::Statement.new(protein_uri,  SIMPATHIC['original-id'], RDF::Literal.new("#{gene_id}"), graph_name: context_uri)
  graph << RDF::Statement.new(protein_uri,  SIMPATHIC['in-taxon'], taxon, graph_name: context_uri)

  
  graph << RDF::Statement.new(context_uri, SIMPATHIC['skg-source'], RDF::Literal.new("Radboud"), graph_name: general_context)
#   graph << RDF::Statement.new(context_uri, SIMPATHIC['evidence'], RDF::URI.new(evidence))
#   graph << RDF::Statement.new(context_uri, SIMPATHIC['score'], RDF::Literal.new(score))


#   warn "graph #{context_uri} built"
  # Write RDF to file in N-Quads format
  File.open('./graph/disease-gene.nq.large', 'a') do |f|
    RDF::Writer.for(:nquads).new(f) do |writer|
#         warn "writing quads"
      writer << graph
    end
  end
#   warn "end graph writing"

end
warn "completed graph building"
e.close

puts "RDF quads written"

(irb):7: warning: already initialized constant Object::SIMPATHIC
(irb):7: warning: previous definition of SIMPATHIC was here
(irb):8: warning: already initialized constant Object::RDFS
(irb):8: warning: previous definition of RDFS was here
disease lookup failed HP_0000857
disease lookup failed HP_0012076
disease lookup failed HP_0000520
disease lookup failed HP_0000802
disease lookup failed HP_0001591
disease lookup failed HP_0002035
disease lookup failed HP_0008909
disease lookup failed HP_0009816
disease lookup failed HP_0010865
disease lookup failed HP_0012042
disease lookup failed HP_0002571
disease lookup failed HP_0004308
disease lookup failed HP_0012390
disease lookup failed HP_0012410
disease lookup failed HP_0100727
disease lookup failed HP_0000327
disease lookup failed HP_0006789
disease lookup failed HP_0030834
disease lookup failed EFO_0003073
disease lookup failed HP_0000200
disease lookup failed HP_0002904
disease lookup failed HP_0002153
disease lookup failed HP_0000876


RDF quads written
